In [ ]:
import pandas as pd

# path constants
DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

# load eye data
baseline_eye_tracking = pd.read_csv(f'{DATA}/sed.csv')
eye_tracking_01 = pd.read_csv(f'{DATA}/sed_01.csv')
eye_tracking_02 = pd.read_csv(f'{DATA}/sed_02.csv')
eye_tracking_03 = pd.read_csv(f'{DATA}/sed_03.csv')

# load psychometric data
psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

# rename columns
columns_mapping = {
    'datetime': 'timestamp',
    'pupil': 'pupil_dilation',
    'leftEyeOpen': 'left_blink',
    'rightEyeOpen': 'right_blink'
}
baseline_eye_tracking = baseline_eye_tracking.rename(columns=columns_mapping)
eye_tracking_01 = eye_tracking_01.rename(columns=columns_mapping)
eye_tracking_02 = eye_tracking_02.rename(columns=columns_mapping)
eye_tracking_03 = eye_tracking_03.rename(columns=columns_mapping)

# clean eye timestamps
def clean_eye_tracking_data(eye_tracking_data):
    eye_tracking_data['timestamp'] = pd.to_datetime(
        eye_tracking_data['timestamp'], utc=True, errors='coerce'
    ).dt.tz_convert(None)
    eye_tracking_data = eye_tracking_data.dropna(subset=['timestamp'])
    return eye_tracking_data

baseline_eye_tracking = clean_eye_tracking_data(baseline_eye_tracking)
eye_tracking_01 = clean_eye_tracking_data(eye_tracking_01)
eye_tracking_02 = clean_eye_tracking_data(eye_tracking_02)
eye_tracking_03 = clean_eye_tracking_data(eye_tracking_03)

# clean psychometric timestamps
for _psy in (psychometric_01, psychometric_02, psychometric_03):
    _psy['Question Start Time'] = pd.to_datetime(
        _psy['Question Start Time'], utc=True, errors='coerce'
    ).dt.tz_convert(None)
    _psy['Question Answer Time'] = pd.to_datetime(
        _psy['Question Answer Time'], utc=True, errors='coerce'
    ).dt.tz_convert(None)
psychometric_01 = psychometric_01.dropna(subset=['Question Start Time'])
psychometric_02 = psychometric_02.dropna(subset=['Question Start Time'])
psychometric_03 = psychometric_03.dropna(subset=['Question Start Time'])

# filter by time window
def filter_eye_tracking_data(eye_tracking_data, question):
    start_time = question['Question Start Time']
    end_time = question['Question Answer Time']
    return eye_tracking_data[
        (eye_tracking_data['timestamp'] >= start_time) &
        (eye_tracking_data['timestamp'] <= end_time)
    ]

# compute eye metrics
def calculate_eye_tracking_metrics(eye_tracking_data):
    average_pupil_dilation = eye_tracking_data['pupil_dilation'].mean()
    blink_detection_threshold = 1.0
    left_blink_count = (
        (eye_tracking_data['left_blink'] > blink_detection_threshold) &
        (eye_tracking_data['left_blink'].shift(-1) <= blink_detection_threshold)
    ).sum()
    right_blink_count = (
        (eye_tracking_data['right_blink'] > blink_detection_threshold) &
        (eye_tracking_data['right_blink'].shift(-1) <= blink_detection_threshold)
    ).sum()
    total_duration_minutes = (
        (eye_tracking_data['timestamp'].max() - eye_tracking_data['timestamp'].min())
        .total_seconds() / 60
    )
    left_blink_rate = left_blink_count / total_duration_minutes
    right_blink_rate = right_blink_count / total_duration_minutes
    return average_pupil_dilation, left_blink_rate, right_blink_rate

# baseline metrics
baseline_metrics = calculate_eye_tracking_metrics(baseline_eye_tracking)

# detect significant increase
def detect_significant_increase(test_metrics, baseline_metrics):
    pupil_dilation_increase = test_metrics[0] > baseline_metrics[0]
    left_blink_rate_increase = test_metrics[1] > baseline_metrics[1]
    right_blink_rate_increase = test_metrics[2] > baseline_metrics[2]
    return pupil_dilation_increase, left_blink_rate_increase, right_blink_rate_increase

# filter correct questions
def filter_correct_questions(df, types_count):
    filtered_df = pd.DataFrame()
    for q_type, count in types_count.items():
        filtered_df = pd.concat([filtered_df, df[df['Type'] == q_type].head(count)])
    return filtered_df

In [ ]:
# Filter for HADS questions only
questions_hads_01 = psychometric_01[psychometric_01['Type'] == 'HADS'].copy()
questions_hads_02 = psychometric_02[psychometric_02['Type'] == 'HADS'].copy()
questions_hads_03 = psychometric_03[psychometric_03['Type'] == 'HADS'].copy()

# Calculate metrics for HADS (simple version, no baseline comparison)
def calculate_metrics_for_hads(questions, eye_tracking_data):
    results = []
    for _, question in questions.iterrows():
        filtered_data = filter_eye_tracking_data(eye_tracking_data, question)
        if not filtered_data.empty:
            metrics = calculate_eye_tracking_metrics(filtered_data)
            results.append({
                'Question': question['Question'],
                'Start Time': question['Question Start Time'],
                'End Time': question['Question Answer Time'],
                'Score': question['Answer'],
                'Average Pupil Dilation': metrics[0],
                'Average Left Blink Rate': metrics[1],
                'Average Right Blink Rate': metrics[2]
            })
    return pd.DataFrame(results)

results_01 = calculate_metrics_for_hads(questions_hads_01, eye_tracking_01)
results_02 = calculate_metrics_for_hads(questions_hads_02, eye_tracking_02)
results_03 = calculate_metrics_for_hads(questions_hads_03, eye_tracking_03)

results_01, results_02, results_03

In [ ]:
# Filter for all relevant question types
types = ['HADS', 'STAI-S', 'STAI-T', 'BFI', 'FQ']
questions_01 = psychometric_01[psychometric_01['Type'].isin(types)].copy()
questions_02 = psychometric_02[psychometric_02['Type'].isin(types)].copy()
questions_03 = psychometric_03[psychometric_03['Type'].isin(types)].copy()

# Calculate metrics (with Type, no baseline comparison)
def calculate_metrics_for_questions(questions, eye_tracking_data):
    results = []
    for _, question in questions.iterrows():
        filtered_data = filter_eye_tracking_data(eye_tracking_data, question)
        if not filtered_data.empty:
            metrics = calculate_eye_tracking_metrics(filtered_data)
            results.append({
                'Type': question['Type'],
                'Question': question['Question'],
                'Start Time': question['Question Start Time'],
                'End Time': question['Question Answer Time'],
                'Score': question['Answer'],
                'Average Pupil Dilation': metrics[0],
                'Average Left Blink Rate': metrics[1],
                'Average Right Blink Rate': metrics[2]
            })
    return pd.DataFrame(results)

results_01 = calculate_metrics_for_questions(questions_01, eye_tracking_01)
results_02 = calculate_metrics_for_questions(questions_02, eye_tracking_02)
results_03 = calculate_metrics_for_questions(questions_03, eye_tracking_03)

# Combine and save
results_01['Test'] = 'Test 01'
results_02['Test'] = 'Test 02'
results_03['Test'] = 'Test 03'

combined_results = pd.concat([results_01, results_02, results_03], ignore_index=True)
combined_results.to_csv(f'{DATA}/QQ.csv', index=False)

In [ ]:
# Filter for all relevant question types
types = ['HADS', 'STAI-S', 'STAI-T', 'BFI', 'FQ']
questions_01 = psychometric_01[psychometric_01['Type'].isin(types)].copy()
questions_02 = psychometric_02[psychometric_02['Type'].isin(types)].copy()
questions_03 = psychometric_03[psychometric_03['Type'].isin(types)].copy()

# Calculate metrics with baseline comparison (anxiety detection)
def calculate_metrics_with_baseline(questions, eye_tracking_data, baseline_metrics):
    results = []
    for _, question in questions.iterrows():
        filtered_data = filter_eye_tracking_data(eye_tracking_data, question)
        if not filtered_data.empty:
            metrics = calculate_eye_tracking_metrics(filtered_data)
            significant_increases = detect_significant_increase(metrics, baseline_metrics)
            results.append({
                'Type': question['Type'],
                'Question': question['Question'],
                'Start Time': question['Question Start Time'],
                'End Time': question['Question Answer Time'],
                'Score': question['Answer'],
                'Average Pupil Dilation': metrics[0],
                'Average Left Blink Rate': metrics[1],
                'Average Right Blink Rate': metrics[2],
                'Sign of Anxiety': 'Yes' if any(significant_increases) else 'No'
            })
    return pd.DataFrame(results)

results_01 = calculate_metrics_with_baseline(questions_01, eye_tracking_01, baseline_metrics)
results_02 = calculate_metrics_with_baseline(questions_02, eye_tracking_02, baseline_metrics)
results_03 = calculate_metrics_with_baseline(questions_03, eye_tracking_03, baseline_metrics)

# Combine and save
results_01['Test'] = 'Test 01'
results_02['Test'] = 'Test 02'
results_03['Test'] = 'Test 03'

combined_results = pd.concat([results_01, results_02, results_03], ignore_index=True)
combined_results.to_csv(f'{DATA}/QQ.csv', index=False)

combined_results.head()

In [ ]:
# Validate – correcting total number of questions per type
types_count = {
    'HADS': 14,
    'STAI-S': 20,
    'STAI-T': 20,
    'BFI': 10,
    'FQ': 24
}

questions_01 = filter_correct_questions(psychometric_01, types_count)
questions_02 = filter_correct_questions(psychometric_02, types_count)
questions_03 = filter_correct_questions(psychometric_03, types_count)

# Calculate metrics with baseline comparison (rounded, with per-metric flags)
def calculate_metrics_validated(questions, eye_tracking_data, baseline_metrics):
    results = []
    for _, question in questions.iterrows():
        filtered_data = filter_eye_tracking_data(eye_tracking_data, question)
        if not filtered_data.empty:
            metrics = calculate_eye_tracking_metrics(filtered_data)
            significant_increases = detect_significant_increase(metrics, baseline_metrics)
            results.append({
                'Type': question['Type'],
                'Question': question['Question'],
                'Start Time': question['Question Start Time'],
                'End Time': question['Question Answer Time'],
                'Score': question['Answer'],
                'Average Pupil Dilation': round(metrics[0], 2),
                'Average Left Blink Rate': round(metrics[1], 2),
                'Average Right Blink Rate': round(metrics[2], 2),
                'Sign of Anxiety': 'Yes' if any(significant_increases) else 'No',
                'Pupil Dilation Increase': 'Yes' if significant_increases[0] else 'No',
                'Left Blink Rate Increase': 'Yes' if significant_increases[1] else 'No',
                'Right Blink Rate Increase': 'Yes' if significant_increases[2] else 'No'
            })
    return pd.DataFrame(results)

results_01 = calculate_metrics_validated(questions_01, eye_tracking_01, baseline_metrics)
results_02 = calculate_metrics_validated(questions_02, eye_tracking_02, baseline_metrics)
results_03 = calculate_metrics_validated(questions_03, eye_tracking_03, baseline_metrics)

# Combine and validate row count
results_01['Test'] = 'Test 01'
results_02['Test'] = 'Test 02'
results_03['Test'] = 'Test 03'

combined_results = pd.concat([results_01, results_02, results_03], ignore_index=True)

expected_rows = sum(types_count.values()) * 3
assert len(combined_results) == expected_rows, f"Expected {expected_rows} rows, but got {len(combined_results)}"

# Save data
combined_results.to_csv(f'{DATA}/QQ2.csv', index=False)

combined_results.head()